# Scraper SportManiacs

Descarga del catalogo de carreras de sportmaniacs.com, ficha de detalle de cada una y recuento de participantes.

Todas las rutas de fichero de este notebook son relativas a la carpeta donde vive (`sportmaniacs_data`); ejecutalo con esa carpeta como directorio de trabajo.

La limpieza de los datos ya descargados (duplicados, telefono/email, filtro por pais...) vive en el notebook separado `limpieza_sportmaniacs.ipynb`, no aqui.

## 1. Catalogo de carreras

Endpoint que devuelve todas las carreras del sitio como pares `slug` / `nombre`.

In [1]:
import requests

url = "https://api-aws.sportmaniacs.com/api/races?prefetch=true&lang=es"

r = requests.get(url)

print("Status:", r.status_code)
print("Claves del JSON:", r.json().keys())
print("Numero de carreras:", len(r.json()["data"]))

Status: 200
Claves del JSON: dict_keys(['data', 'status'])
Numero de carreras: 24651


In [2]:
import json
from pathlib import Path

import requests

BASE = Path("../../data/raw/sportmaniacs")

URL = "https://api-aws.sportmaniacs.com/api/races?prefetch=true&lang=es"

r = requests.get(URL)
r.raise_for_status()

ruta = BASE / "sportmaniacs_races.json"

with open(ruta, "w", encoding="utf-8") as f:
    json.dump(r.json(), f, ensure_ascii=False, indent=2)

print(f"Fichero guardado en {ruta}")

Fichero guardado en ../../data/raw/sportmaniacs/sportmaniacs_races.json


In [3]:
import json

import pandas as pd

with open(BASE / "sportmaniacs_races.json", encoding="utf-8") as f:
    data = json.load(f)

df = pd.DataFrame(data["data"])

print(df.head())
print(f"\nTotal de carreras: {len(df)}")

df.to_csv(BASE / "sportmaniacs_catalogo_carreras.csv", index=False, encoding="utf-8-sig")

print("CSV creado.")

                                      key  \
0             san-siltrail-2024-guatemala   
1                 tarraco-healthrace-2024   
2                   revolution-hybrid-run   
3           cursa-popular-san-agusti-2026   
4  2025-2026-gxcl-match-2-llandaff-fields   

                                            value  
0                     San Siltrail 2024 Guatemala  
1                         TARRACO HEALTHRACE 2024  
2                           Revolution Hybrid Run  
3  Cursa popular Sant Agustí · MusicRunNight 2026  
4          2025-2026 GXCL Match 2 Llandaff Fields  

Total de carreras: 24651
CSV creado.


## 2. Exploracion de la API de detalle

Comprobacion de que la ruta correcta es `/api/races/{slug}` (las variantes `/race/`, `/events/`, `/event/` no funcionan).

In [4]:
import requests

slug = "xxii-cursa-del-riu-ripoll-de-sabadell"

posibles = [
    f"https://api-aws.sportmaniacs.com/api/races/{slug}",
    f"https://api-aws.sportmaniacs.com/api/race/{slug}",
    f"https://api-aws.sportmaniacs.com/api/events/{slug}",
    f"https://api-aws.sportmaniacs.com/api/event/{slug}",
]

for url in posibles:
    try:
        r = requests.get(url, timeout=10)
        print("\n", url)
        print("Status:", r.status_code)
        print(r.text[:300])
    except Exception as e:
        print(e)


 https://api-aws.sportmaniacs.com/api/races/xxii-cursa-del-riu-ripoll-de-sabadell
Status: 200
{"data":{"id":"048630df-f558-4fd7-ae38-d49317cb7294","name":"XXII Cursa del riu Ripoll de Sabadell","idRace":"000650000200176","date":"2024-12-26","end_date":"2024-12-26","slug":"xxii-cursa-del-riu-ripoll-de-sabadell","idRaceType":"0","province_id":"8","province":"Barcelona","country_id":"ESP","coun

 https://api-aws.sportmaniacs.com/api/race/xxii-cursa-del-riu-ripoll-de-sabadell
Status: 404
<!DOCTYPE html>
	<html lang="en">
	<head>
		<meta charset="UTF-8">
		<title>Sportmaniacs - Opps, ha ocurrido un error </title>

		<style>

			html {
				width: 100%;
				height: 100%;
				overflow: hidden;
			}

			body {
				width: 100%;
				height: 100%;
				overflow: hidden;
				background: url('

 https://api-aws.sportmaniacs.com/api/events/xxii-cursa-del-riu-ripoll-de-sabadell
Status: 200
{"status":"ko","message":""}

 https://api-aws.sportmaniacs.com/api/event/xxii-cursa-del-riu-ripoll-de-sabadell
S

In [5]:
import requests

slug = "xi-mitja-de-tortosa-amp-10k"

info = requests.get(f"https://api-aws.sportmaniacs.com/api/races/{slug}").json()["data"]

print(info.keys())

dict_keys(['id', 'name', 'idRace', 'date', 'end_date', 'slug', 'idRaceType', 'province_id', 'province', 'country_id', 'country', 'city_id', 'city', 'dev_info', 'hour_info', 'dorsal_info', 'latitude', 'longitude', 'photos', 'header', 'poster', 'description', 'status', 'active_photos', 'active_credits', 'previous_race', 'next_race', 'showRankings', 'externalInscriptions', 'timezone', 'validatePhotos', 'files', 'contact_info', 'legal_advice', 'rules', 'rules_link', 'rules_file'])


## 3. Descarga del detalle de cada carrera

Recorre el catalogo y guarda los campos relevantes de cada carrera en `sportmaniacs_curses_complet.csv`. Resumible: si el fichero de salida ya existe, continua desde donde se quedo en vez de volver a empezar.

In [6]:
from pathlib import Path
import time

import pandas as pd
import requests

BASE = Path("../../data/raw/sportmaniacs")
INPUT = BASE / "sportmaniacs_catalogo_carreras.csv"
OUTPUT = BASE / "sportmaniacs_curses_complet.csv"

df = pd.read_csv(INPUT)

if OUTPUT.exists():
    df_final = pd.read_csv(OUTPUT)
    descargadas = set(df_final["slug"].astype(str))
    resultados = df_final.to_dict("records")
    print(f"Se reanuda la descarga. Ya hay {len(resultados)} carreras.")
else:
    descargadas = set()
    resultados = []
    print("Empezando desde cero.")

TOTAL = len(df)

for i, row in df.iterrows():
    slug = str(row["key"])

    if slug in descargadas:
        continue

    print(f"[{i+1}/{TOTAL}] {slug}")

    url = f"https://api-aws.sportmaniacs.com/api/races/{slug}"

    try:
        r = requests.get(url, timeout=20)

        if r.status_code != 200:
            print(f"  Error {r.status_code}")
            continue

        info = r.json()["data"]
        contacto = info.get("contact_info") or {}
        ficheros = info.get("files") or []

        resultados.append({
            "slug": info.get("slug"),
            "id": info.get("id"),
            "idRace": info.get("idRace"),
            "nom": info.get("name"),
            "data": info.get("date"),
            "data_final": info.get("end_date"),
            "ciutat": info.get("city"),
            "provincia": info.get("province"),
            "pais": info.get("country"),
            "latitude": info.get("latitude"),
            "longitude": info.get("longitude"),
            "descripcio": info.get("description"),
            "email": contacto.get("email"),
            "telefon": contacto.get("phone"),
            "reglament": info.get("rules_file"),
            "num_fitxers": len(ficheros),
            "showRankings": info.get("showRankings"),
            "active_photos": info.get("active_photos"),
            "externalInscriptions": info.get("externalInscriptions"),
        })

        descargadas.add(slug)

        if len(resultados) % 100 == 0:
            pd.DataFrame(resultados).to_csv(OUTPUT, index=False, encoding="utf-8-sig")
            print(f"  Guardadas {len(resultados)} carreras.")

        time.sleep(0.25)

    except Exception as e:
        print(f"  {e}")

pd.DataFrame(resultados).to_csv(OUTPUT, index=False, encoding="utf-8-sig")

print()
print("===================================")
print("PROCESO FINALIZADO")
print(f"Carreras guardadas: {len(resultados)}")
print(f"Fichero: {OUTPUT}")
print("===================================")

Se reanuda la descarga. Ya hay 24363 carreras.
[4/24651] cursa-popular-san-agusti-2026
[61/24651] spike-island-5km-race
[86/24651] carrera-mas-rboles-mas-bosques-2026
[125/24651] iv-triatlon-supersprint-de-la-roda-2026
[188/24651] odessa-autumn-blaze-5k
[192/24651] totalenergies-gran-fondo-alberto-contador-2026
[220/24651] ii-carrera-nocturna-de-segovia-ciudades-patrimonio-de-la-humanidad
[222/24651] the-bay-aquathlon-2026-series-race-3
[229/24651] carrera-nocturna-quismondo-2026
[239/24651] tri-hard-harriers-5k-on-the-bay-2026
[269/24651] swansea-bay-10k-2026
[287/24651] hyrox-mtu-tralee-sim
[305/24651] prorun-x-nightrun
[314/24651] 47-carrera-popular-fiestas-de-la-elipa
[326/24651] topman-y-seaman-concn-2026
[351/24651] carrera-atltica-de-la-contadura-pblica
[366/24651] carrera-dcimo-aniversario-unsa---unsea
[388/24651] heart-challenge-night-edition
[393/24651] v-pujada-a-la-serreta---vedruna
[526/24651] laser-run-school-games-2026
[4182/24651] the-gunflint-scramble
[6327/24651] dai-

## 4. Migracion de un esquema antiguo (ya ejecutada)

Paso puntual, ejecutado una sola vez, para descartar filas y columnas de una version anterior del fichero que no tenia todavia los campos de detalle. Se deja aqui como registro; no hace falta volver a ejecutarlo. Para limpieza continua de los datos (duplicados, telefono/email, filtro por pais...) usa `limpieza_sportmaniacs.ipynb`.

In [7]:
from pathlib import Path

import pandas as pd

OUTPUT = Path("../../data/raw/sportmaniacs") / "sportmaniacs_curses_complet.csv"

df_final = pd.read_csv(OUTPUT)
print(f"Filas antes de limpiar: {len(df_final)}")

# Fuera las filas del esquema antiguo (no tienen campos de detalle)
df_final = df_final[df_final["num_fitxers"].notna()].copy()

# Fuera las columnas muertas del esquema antiguo
columnas_muertas = [
    "data_fi", "adreca", "tipus", "organitzador", "web",
    "facebook", "instagram", "twitter", "imatge", "latitud", "longitud",
]
df_final = df_final.drop(columns=[c for c in columnas_muertas if c in df_final.columns])

df_final.to_csv(OUTPUT, index=False, encoding="utf-8-sig")

print(f"Filas despues de limpiar: {len(df_final)}")
print(f"Columnas: {df_final.columns.tolist()}")

Filas antes de limpiar: 24660
Filas despues de limpiar: 24660
Columnas: ['id', 'idRace', 'nom', 'slug', 'data', 'ciutat', 'provincia', 'pais', 'email', 'telefon', 'descripcio', 'data_final', 'latitude', 'longitude', 'reglament', 'num_fitxers', 'showRankings', 'active_photos', 'externalInscriptions']


## 5. Recuento de participantes por sexo, por carrera

No existe ningun campo de participantes en `/api/races/{slug}`. El dato real esta en la pagina publica de resultados, mediante dos pasos:

1. La pagina HTML normal `https://sportmaniacs.com/es/races/{slug}` incluye enlaces con un UUID por cada prueba/distancia de la carrera, con el patron `/es/races/{slug}/{event_id}/results`. Una carrera puede tener 0 (sin resultados publicados), 1 o varias distancias.
2. Para cada `event_id` se puede pedir `https://sportmaniacs.com/es/races/rankings/{event_id}` (con `Accept: application/json`), que devuelve un JSON con la lista completa de corredores en `data["Rankings"]`. Cada corredor tiene un campo `gender` que vale `gender_0` (hombre), `gender_1` (mujer) o `gender_-1` (no especificado).

**Importante:** el campo `gender` no esta siempre relleno. En algunas carreras (depende de si el organizador/cronometrador lo cargo) todos los corredores salen como `gender_-1`, y el sexo solo se podria intuir por el nombre de la categoria (p. ej. "VETERANOS 35M" / "VETERANAS 35F"), que es texto libre y no uniforme entre carreras (castellano, catalan, euskera...). Por eso el codigo **no intenta adivinar el sexo a partir del nombre de la categoria** — solo cuenta lo que el campo `gender` diga explicitamente, y guarda aparte cuantos quedan sin especificar para que se vea la cobertura real de este dato.

Verificado con casos reales:
- `medio-maraton-critas-del-2026`: 5246 participantes, pero el organizador no cargo el campo `gender` -> los 5246 salen como sin especificar.
- `iii-podoactiva-medieval-trail-montearagn`: 134 participantes, 114 hombres / 20 mujeres (aqui si esta bien relleno).
- `xxii-cursa-del-riu-ripoll-de-sabadell`: 0 participantes (no tiene resultados publicados).

In [8]:
import re

import requests

sesion = requests.Session()


def eventos_de_carrera(slug):
    """Devuelve los UUID de las distancias/pruebas de una carrera a partir de su pagina publica."""
    r = sesion.get(f"https://sportmaniacs.com/es/races/{slug}", timeout=20)
    r.raise_for_status()
    patron = rf"/es/races/{re.escape(slug)}/([a-f0-9-]{{36}})/results"
    return sorted(set(re.findall(patron, r.text)))


def participantes_de_evento(event_id):
    """Devuelve el recuento por sexo de un event_id, o None si no hay rankings."""
    r = sesion.get(
        f"https://sportmaniacs.com/es/races/rankings/{event_id}",
        headers={"Accept": "application/json"},
        timeout=20,
    )
    if r.status_code != 200:
        return None
    data = r.json().get("data") or {}
    rankings = data.get("Rankings") or []
    hombres = sum(1 for x in rankings if x.get("gender") == "gender_0")
    mujeres = sum(1 for x in rankings if x.get("gender") == "gender_1")
    return {
        "distancia": (data.get("Event") or {}).get("name"),
        "total": len(rankings),
        "hombres": hombres,
        "mujeres": mujeres,
        "sin_especificar": len(rankings) - hombres - mujeres,
    }

Prueba rapida con las 3 carreras usadas para verificar el mecanismo, antes de lanzar el proceso completo. Una linea por distancia.

In [9]:
for slug in [
    "medio-maraton-critas-del-2026",
    "iii-podoactiva-medieval-trail-montearagn",
    "xxii-cursa-del-riu-ripoll-de-sabadell",
]:
    event_ids = eventos_de_carrera(slug)
    if not event_ids:
        print(f"{slug}: sin distancias con resultados")
        continue
    for eid in event_ids:
        info = participantes_de_evento(eid)
        if info is None:
            continue
        print(f"{slug} / {info['distancia']}: total={info['total']} hombres={info['hombres']} "
              f"mujeres={info['mujeres']} sin_especificar={info['sin_especificar']}")

medio-maraton-critas-del-2026 / 4K: total=3302 hombres=0 mujeres=0 sin_especificar=3302
medio-maraton-critas-del-2026 / 10K: total=1384 hombres=0 mujeres=0 sin_especificar=1384
medio-maraton-critas-del-2026 / 21K: total=560 hombres=0 mujeres=0 sin_especificar=560
iii-podoactiva-medieval-trail-montearagn / TRAIL 21K: total=54 hombres=49 mujeres=5 sin_especificar=0
iii-podoactiva-medieval-trail-montearagn / TRAIL 9K: total=80 hombres=65 mujeres=15 sin_especificar=0
xxii-cursa-del-riu-ripoll-de-sabadell: sin distancias con resultados


Proceso completo, resumible igual que la descarga de detalle. Guarda **una fila por cada combinacion carrera+distancia** (no se suman las distancias de una misma carrera). Si una carrera no tiene resultados publicados, se guarda igualmente una fila con `distancia=None` para dejar constancia de que ya se comprobo. Usa `sportmaniacs_curses_net.csv` como listado de entrada (todas las carreras limpias, de cualquier pais); cambia `INPUT` si quieres procesar otro listado.

In [10]:
from pathlib import Path
import time

import pandas as pd

BASE = Path("../../data/raw/sportmaniacs")
INPUT = BASE / "sportmaniacs_curses_net.csv"
OUTPUT = BASE / "sportmaniacs_participantes.csv"

df = pd.read_csv(INPUT)

if OUTPUT.exists():
    df_out = pd.read_csv(OUTPUT)
    hechas = set(df_out["slug"].astype(str))
    resultados = df_out.to_dict("records")
    print(f"Se reanuda. Ya hay {len(hechas)} carreras procesadas ({len(resultados)} filas).")
else:
    hechas = set()
    resultados = []
    print("Empezando desde cero.")

TOTAL = len(df)

for i, row in df.iterrows():
    slug = str(row["slug"])

    if slug in hechas:
        continue

    print(f"[{i+1}/{TOTAL}] {slug}")

    try:
        event_ids = eventos_de_carrera(slug)

        if not event_ids:
            resultados.append({
                "slug": slug,
                "event_id": None,
                "distancia": None,
                "participantes_totales": 0,
                "participantes_hombres": 0,
                "participantes_mujeres": 0,
                "participantes_sin_especificar": 0,
            })
        else:
            for eid in event_ids:
                info = participantes_de_evento(eid)
                if info is None:
                    continue
                resultados.append({
                    "slug": slug,
                    "event_id": eid,
                    "distancia": info["distancia"],
                    "participantes_totales": info["total"],
                    "participantes_hombres": info["hombres"],
                    "participantes_mujeres": info["mujeres"],
                    "participantes_sin_especificar": info["sin_especificar"],
                })

    except Exception as e:
        print(f"  Error: {e}")
        resultados.append({
            "slug": slug,
            "event_id": None,
            "distancia": None,
            "participantes_totales": None,
            "participantes_hombres": None,
            "participantes_mujeres": None,
            "participantes_sin_especificar": None,
        })

    hechas.add(slug)

    if len(hechas) % 100 == 0:
        pd.DataFrame(resultados).to_csv(OUTPUT, index=False, encoding="utf-8-sig")
        print(f"  Guardadas {len(hechas)} carreras ({len(resultados)} filas).")

    time.sleep(0.25)

pd.DataFrame(resultados).to_csv(OUTPUT, index=False, encoding="utf-8-sig")

print()
print("===================================")
print("PROCESO FINALIZADO")
print(f"Carreras procesadas: {len(hechas)}")
print(f"Filas guardadas: {len(resultados)}")
print(f"Fichero: {OUTPUT}")
print("===================================")

Se reanuda. Ya hay 22636 carreras procesadas (44114 filas).

PROCESO FINALIZADO
Carreras procesadas: 22636
Filas guardadas: 44114
Fichero: ../../data/raw/sportmaniacs/sportmaniacs_participantes.csv


## 6. Union con la tabla de carreras

Junta `sportmaniacs_participantes.csv` (una fila por carrera+distancia) con `sportmaniacs_curses_net.csv` por `slug`. El resultado tiene una fila por cada distancia de cada carrera, con todos los datos de la carrera repetidos en cada una.

In [11]:
from pathlib import Path

import pandas as pd

BASE = Path("../../data/raw/sportmaniacs")

carreras = pd.read_csv(BASE / "sportmaniacs_curses_net.csv")
participantes = pd.read_csv(BASE / "sportmaniacs_participantes.csv")

final = carreras.merge(participantes, on="slug", how="left")

SALIDA = BASE / "curses_sportmaniacs_con_participantes.csv"
final.to_csv(SALIDA, index=False, encoding="utf-8-sig")

print(f"Filas: {len(final)}")
print(f"Guardado en {SALIDA}")

Filas: 44114
Guardado en ../../data/raw/sportmaniacs/curses_sportmaniacs_con_participantes.csv
